# MSigDB ORA Analysis: pathway coverage across study sizes

**Environment:** `clamp-analyses`

For each CLAMP model (CLAMPfull and CLAMPbase) across all study coverage levels (1%, 5%, 10%, 25%, 50%, 75%, 100%) and seeds (1–3), this notebook:

1. Loads the Z matrix (gene loadings per LV).
2. For each LV, selects the top 1% genes by descending loading as the gene list.
3. Runs `enricher()` per LV using MSigDB (v2026.1) as gene set database and model genes as universe.
4. Stores raw `terms_padj`: the minimum p.adjust per MSigDB term across all LVs (no FDR threshold applied here).
5. Saves per-model RDS caches (`_msigdb.rds`) and summary CSVs. FDR thresholds are applied in `01_bp_coverage_plot.ipynb`.

In [1]:
library(here)
library(dplyr)
library(clusterProfiler)
library(BiocParallel)

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




clusterProfiler v4.14.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

T Wu, E Hu, S Xu, M Chen, P Guo, Z Dai, T Feng, L Zhou, W Tang, L Zhan,
X Fu, S Liu, X Bo, and G Yu. clusterProfiler 4.0: A universal
enrichment tool for interpreting omics data. The Innovation. 2021,
2(3):100141


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter




## Paths

In [2]:
models_dir <- here("output/01_model_building/04_archs4/07_bp_coverage_study")
output_dir <- here("output/03_model_biology/00_archs4/00_pathway_coverage_bp_study/00_bp_ora_analysis")

dir.create(file.path(output_dir, "CLAMPfull"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(output_dir, "CLAMPbase"), recursive = TRUE, showWarnings = FALSE)

## Model specs: discovered dynamically from disk

In [3]:
study_subdirs <- list.dirs(models_dir, recursive = FALSE, full.names = TRUE)

model_specs <- do.call(c, lapply(study_subdirs, function(study_dir) {
  seed_dirs <- list.dirs(study_dir, recursive = FALSE, full.names = TRUE)
  specs <- lapply(seed_dirs, function(seed_dir) {
    bn <- basename(seed_dir)
    m  <- regmatches(bn, regexec("^study_coverage_rs([0-9]+)_seed_([0-9]+)$", bn))[[1]]
    if (length(m) < 3) return(NULL)
    z_path <- file.path(seed_dir, "CLAMPfull_hall", "Z.csv")
    if (!file.exists(z_path)) return(NULL)
    list(
      coverage = as.integer(m[2]),
      dir      = basename(study_dir),
      seed     = as.integer(m[3]),
      seed_dir = seed_dir
    )
  })
  Filter(Negate(is.null), specs)
}))

model_specs <- model_specs[order(
  sapply(model_specs, `[[`, "coverage"),
  sapply(model_specs, `[[`, "seed")
)]

message("Found ", length(model_specs), " models:")
for (s in model_specs) {
  message(sprintf("  rs%d%% seed%d: %s", s$coverage, s$seed, s$dir))
}

Found 12 models:

  rs1% seed1: 00_bp_coverage_study_01

  rs1% seed2: 00_bp_coverage_study_01

  rs1% seed3: 00_bp_coverage_study_01

  rs5% seed1: 01_bp_coverage_study_05

  rs5% seed2: 01_bp_coverage_study_05

  rs5% seed3: 01_bp_coverage_study_05

  rs10% seed1: 02_bp_coverage_study_10

  rs10% seed2: 02_bp_coverage_study_10

  rs10% seed3: 02_bp_coverage_study_10

  rs25% seed1: 03_bp_coverage_study_25

  rs25% seed2: 03_bp_coverage_study_25

  rs25% seed3: 03_bp_coverage_study_25



## Load MSigDB gene sets

In [4]:
msig_gmt <- clusterProfiler::read.gmt(here("data/pathways/msigdb.v2026.1.Hs.symbols.gmt"))
message(sprintf("MSigDB gene sets loaded: %d", length(unique(msig_gmt$term))))

MSigDB gene sets loaded: 35361



## Helper: run ORA for one model

Returns a list with significant terms and coverage metrics using MSigDB gene sets.

In [ ]:
run_ora_for_model <- function(z_path, rds_path, n_cores = 4) {
  Z              <- read.csv(z_path, row.names = 1, check.names = FALSE)
  universe_genes <- rownames(Z)
  n_top          <- ceiling(0.01 * nrow(Z))
  n_lvs          <- ncol(Z)

  term_overlap   <- tapply(msig_gmt$gene %in% universe_genes, msig_gmt$term, sum)
  n_total_msigdb <- sum(term_overlap >= 10L)

  top_genes_per_lv <- apply(Z, 2, function(lv) {
    universe_genes[order(lv, decreasing = TRUE)[seq_len(n_top)]]
  })

  n_samples <- NA_integer_
  if (file.exists(rds_path)) {
    rds_obj <- readRDS(rds_path)
    if (!is.null(rds_obj$B)) n_samples <- ncol(rds_obj$B)
    rm(rds_obj)
  }

  bp <- BiocParallel::MulticoreParam(workers = n_cores, progressbar = FALSE)
  ora_results <- BiocParallel::bplapply(
    seq_len(n_lvs),
    function(i) {
      genes <- top_genes_per_lv[, i]
      tryCatch(
        clusterProfiler::enricher(
          gene          = genes,
          universe      = universe_genes,
          TERM2GENE     = msig_gmt,
          pAdjustMethod = "BH",
          pvalueCutoff  = 1,
          qvalueCutoff  = 1,
          minGSSize     = 10,
          maxGSSize     = 50000
        ),
        error = function(e) NULL
      )
    },
    BPPARAM = bp
  )

  all_dfs <- lapply(ora_results, function(r) {
    if (is.null(r) || nrow(as.data.frame(r)) == 0) return(NULL)
    as.data.frame(r)
  })
  all_dfs <- Filter(Negate(is.null), all_dfs)

  if (length(all_dfs) == 0) {
    warning("No ORA results returned for: ", z_path)
    return(NULL)
  }

  combined <- do.call(rbind, all_dfs)

  list(
    n_samples      = n_samples,
    n_lvs          = n_lvs,
    n_top_genes    = n_top,
    n_total_msigdb = n_total_msigdb,
    terms_padj     = tapply(combined$p.adjust, combined$ID, min)
  )
}

: 

## Run ORA: CLAMPfull

In [ ]:
coverage_values <- sort(unique(sapply(model_specs, `[[`, "coverage")))
results_clampfull_by_pct <- list()

for (cov in coverage_values) {
  cov_specs    <- Filter(function(s) s$coverage == cov, model_specs)
  pct_rds_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%d_msigdb.rds", cov))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPfull %d%%", cov))
    results_clampfull_by_pct[[as.character(cov)]] <- readRDS(pct_rds_path)
    next
  }

  cov_rows <- lapply(cov_specs, function(spec) {
    z_path     <- file.path(spec$seed_dir, "CLAMPfull_hall", "Z.csv")
    rds_path   <- file.path(spec$seed_dir, "CLAMPfull_hall.rds")
    cache_path <- file.path(output_dir, "CLAMPfull",
                            sprintf("rs%d_seed%d_msigdb.rds", spec$coverage, spec$seed))

    sub_info_path <- file.path(spec$seed_dir, "subsample_info.rds")
    n_studies <- NA_integer_
    if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPfull rs%d seed%d", spec$coverage, spec$seed))
      res <- readRDS(cache_path)
      if (is.null(res$n_studies)) res$n_studies <- n_studies
    } else {
      message(sprintf("Running ORA: CLAMPfull rs%d seed%d", spec$coverage, spec$seed))
      res <- run_ora_for_model(z_path, rds_path)
      if (!is.null(res)) {
        res$n_studies <- n_studies
        saveRDS(res, cache_path)
      }
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPfull",
      coverage_pct   = spec$coverage,
      seed           = spec$seed,
      n_studies      = res$n_studies,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), cov_rows))
  rownames(pct_df) <- NULL
  results_clampfull_by_pct[[as.character(cov)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPfull %d%% -> %s", cov, pct_rds_path))
}

results_clampfull_df <- do.call(rbind, results_clampfull_by_pct)
rownames(results_clampfull_df) <- NULL
print(results_clampfull_df)

Running ORA: CLAMPfull rs1 seed1

Running ORA: CLAMPfull rs1 seed2

Running ORA: CLAMPfull rs1 seed3

Saved: CLAMPfull 1% -> /home/msubirana/Documents/pivlab/clamp-analyses/output/03_model_biology/00_archs4/00_pathway_coverage_bp_study/00_bp_ora_analysis/CLAMPfull/results_pct1_msigdb.rds

Running ORA: CLAMPfull rs5 seed1

Running ORA: CLAMPfull rs5 seed2



In [ ]:
for (cov in names(results_clampfull_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPfull", sprintf("results_pct%s_msigdb.csv", cov))
  write.csv(results_clampfull_by_pct[[cov]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}

## Run ORA: CLAMPbase

In [ ]:
results_clampbase_by_pct <- list()

for (cov in coverage_values) {
  cov_specs    <- Filter(function(s) s$coverage == cov, model_specs)
  pct_rds_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%d_msigdb.rds", cov))

  if (file.exists(pct_rds_path)) {
    message(sprintf("Loading cached pct-level result: CLAMPbase %d%%", cov))
    results_clampbase_by_pct[[as.character(cov)]] <- readRDS(pct_rds_path)
    next
  }

  cov_rows <- lapply(cov_specs, function(spec) {
    z_path     <- file.path(spec$seed_dir, "CLAMPbase", "Z.csv")
    rds_path   <- file.path(spec$seed_dir, "CLAMPbase.rds")
    cache_path <- file.path(output_dir, "CLAMPbase",
                            sprintf("rs%d_seed%d_msigdb.rds", spec$coverage, spec$seed))

    sub_info_path <- file.path(spec$seed_dir, "subsample_info.rds")
    n_studies <- NA_integer_
    if (file.exists(sub_info_path)) n_studies <- readRDS(sub_info_path)$n_studies

    if (!file.exists(z_path)) { warning("Z.csv not found: ", z_path); return(NULL) }

    if (file.exists(cache_path)) {
      message(sprintf("Loading cached: CLAMPbase rs%d seed%d", spec$coverage, spec$seed))
      res <- readRDS(cache_path)
      if (is.null(res$n_studies)) res$n_studies <- n_studies
    } else {
      message(sprintf("Running ORA: CLAMPbase rs%d seed%d", spec$coverage, spec$seed))
      res <- run_ora_for_model(z_path, rds_path)
      if (!is.null(res)) {
        res$n_studies <- n_studies
        saveRDS(res, cache_path)
      }
    }
    if (is.null(res)) return(NULL)

    data.frame(
      model_type     = "CLAMPbase",
      coverage_pct   = spec$coverage,
      seed           = spec$seed,
      n_studies      = res$n_studies,
      n_samples      = res$n_samples,
      n_lvs          = res$n_lvs,
      n_top_genes    = res$n_top_genes,
      n_total_msigdb = res$n_total_msigdb,
      stringsAsFactors = FALSE
    )
  })

  pct_df <- do.call(rbind, Filter(Negate(is.null), cov_rows))
  rownames(pct_df) <- NULL
  results_clampbase_by_pct[[as.character(cov)]] <- pct_df
  saveRDS(pct_df, pct_rds_path)
  message(sprintf("Saved: CLAMPbase %d%% -> %s", cov, pct_rds_path))
}

results_clampbase_df <- do.call(rbind, results_clampbase_by_pct)
rownames(results_clampbase_df) <- NULL
print(results_clampbase_df)

In [ ]:
for (cov in names(results_clampbase_by_pct)) {
  csv_path <- file.path(output_dir, "CLAMPbase", sprintf("results_pct%s_msigdb.csv", cov))
  write.csv(results_clampbase_by_pct[[cov]], csv_path, row.names = FALSE)
  message("Saved: ", csv_path)
}